# Requirement 3: Primal-Dual on Shared Non-Stationary Realizations

This notebook evaluates the factorized primal-dual Hedge learner on the exact same precomputed market matrices used by Requirement 4. The shared inputs make the comparison with sliding-window CUCB and change-detector CUCB reproducible and fair.

## 1. Setup

The experiment contains 3 symmetric-KL levels (20, 200, 2000), 3 change frequencies (1/100, 1/50, 1/25), and 10 realizations per setting.

In [ ]:
from pathlib import Path
import csv
import json
import sys

import matplotlib.pyplot as plt
import numpy as np

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import experiments.run_req3_on_shared_realizations as shared_experiment

MATRIX_DIR = PROJECT_ROOT / 'results' / 'processed' / 'req4_shared_realizations'
RESULT_DIR = PROJECT_ROOT / 'results' / 'processed' / 'req3_on_req4_shared'
print('Project root:', PROJECT_ROOT)
print('Matrix directory:', MATRIX_DIR)

## 2. Validate the Shared Matrices

Every saved realization must be a finite `3000 x 10` matrix with entries in `[0, 1]`. Each setting also stores the constellation, random seeds, change points, and active mode schedule.

In [ ]:
settings = sorted(path for path in MATRIX_DIR.iterdir() if path.is_dir())
matrix_paths = sorted(MATRIX_DIR.glob('*/realization_*.npy'))

assert len(settings) == 9, f'Expected 9 settings, found {len(settings)}'
assert len(matrix_paths) == 90, f'Expected 90 matrices, found {len(matrix_paths)}'

for path in matrix_paths:
    matrix = np.load(path, mmap_mode='r')
    assert matrix.shape == (shared_experiment.T, shared_experiment.N)
    assert np.isfinite(matrix).all()
    assert matrix.min() >= 0.0 and matrix.max() <= 1.0

print(f'Validated {len(matrix_paths)} matrices across {len(settings)} settings.')

## 3. Inspect One Market Realization

The example below uses medium sKL and medium change frequency. Rows represent rounds and columns represent campaigns.

In [ ]:
example_setting = MATRIX_DIR / 'medium_frequency_medium_skl'
example_path = example_setting / 'realization_00.npy'
example_config = json.loads((example_setting / 'config.json').read_text(encoding='utf-8'))
example_matrix = np.load(example_path)

print('Target sKL:', example_config['target_skl'])
print('Change frequency:', example_config['change_frequency'])
print('Number of changes:', len(example_config['change_points']))
print('Matrix shape:', example_matrix.shape)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].imshow(example_matrix.T, aspect='auto', cmap='viridis', vmin=0, vmax=1)
axes[0].set_title('Highest competing bids')
axes[0].set_xlabel('Round')
axes[0].set_ylabel('Campaign')
for campaign in range(min(4, example_matrix.shape[1])):
    axes[1].plot(example_matrix[:, campaign], alpha=0.8, label=f'Campaign {campaign}')
axes[1].set_title('Example campaign streams')
axes[1].set_xlabel('Round')
axes[1].set_ylabel('Competing bid')
axes[1].legend()
plt.tight_layout()
plt.show()

## 4. Run the Factorized Primal-Dual Learner

The learner maintains one Hedge distribution for each campaign and updates a dual multiplier to control total expenditure. The environment below is backed directly by the saved matrix, so no market randomness is regenerated during the run.

In [ ]:
trial = shared_experiment.run_factorized_primal_dual_trial(
    env_seed=0,
    learner_seed=2000,
    T=shared_experiment.T,
    N=shared_experiment.N,
    K_value=int(example_config['oracle_block_length']),
    bid_set=shared_experiment.BID_SET,
    values=shared_experiment.VALUES,
    budget=shared_experiment.BUDGET,
    environment='precomputed',
    environment_kwargs={'path': example_path},
)

print(f"Final utility: {trial['cumulative_utility'][-1]:.2f}")
print(f"Final cost: {trial['cumulative_cost'][-1]:.2f}")
print(f"Budget left: {trial['remaining_budgets'][-1]:.2f}")

In [ ]:
rounds = np.arange(1, shared_experiment.T + 1)
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes[0, 0].plot(rounds, trial['cumulative_utility'])
axes[0, 0].set_title('Cumulative utility')
axes[0, 1].plot(rounds, trial['cumulative_cost'])
axes[0, 1].axhline(shared_experiment.BUDGET, color='black', linestyle='--', label='Budget')
axes[0, 1].set_title('Cumulative cost')
axes[0, 1].legend()
axes[1, 0].plot(rounds, trial['remaining_budgets'])
axes[1, 0].set_title('Budget left')
axes[1, 1].plot(rounds, trial['lambdas'])
axes[1, 1].set_title('Dual multiplier')
for ax in axes.flat:
    ax.set_xlabel('Round')
    ax.grid(alpha=0.2)
plt.tight_layout()
plt.show()

## 5. Run All 90 Experiments

Set `RUN_ALL = True` when results must be regenerated. The runner saves one row per realization and a 9-row aggregate summary.

In [ ]:
RUN_ALL = False
if RUN_ALL:
    shared_experiment.main()
else:
    print('Using the saved results. Set RUN_ALL = True to regenerate all 90 runs.')

## 6. Aggregate Results

The following table and plots summarize mean utility and cost over the 10 shared realizations in every sKL/frequency combination.

In [ ]:
summary_path = RESULT_DIR / 'req3_factorized_summary.csv'
with summary_path.open(newline='', encoding='utf-8') as handle:
    summary = list(csv.DictReader(handle))

columns = ['frequency_label', 'skl_label', 'n_realizations', 'utility_mean', 'utility_std', 'cost_mean']
print(' | '.join(columns))
print('-' * 105)
for row in summary:
    print(' | '.join(row[column] for column in columns))

In [ ]:
frequency_order = ['low_frequency', 'medium_frequency', 'high_frequency']
skl_order = ['low_skl', 'medium_skl', 'high_skl']
utility_grid = np.zeros((3, 3))
cost_grid = np.zeros((3, 3))
for row in summary:
    i = frequency_order.index(row['frequency_label'])
    j = skl_order.index(row['skl_label'])
    utility_grid[i, j] = float(row['utility_mean'])
    cost_grid[i, j] = float(row['cost_mean'])

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, grid, title in zip(axes, [utility_grid, cost_grid], ['Mean final utility', 'Mean final cost']):
    image = ax.imshow(grid, aspect='auto', cmap='Blues')
    ax.set_xticks(range(3), ['sKL 20', 'sKL 200', 'sKL 2000'])
    ax.set_yticks(range(3), ['1/100', '1/50', '1/25'])
    ax.set_xlabel('Symmetric KL')
    ax.set_ylabel('Change frequency')
    ax.set_title(title)
    for i in range(3):
        for j in range(3):
            ax.text(j, i, f'{grid[i, j]:.1f}', ha='center', va='center')
    fig.colorbar(image, ax=ax, shrink=0.8)
plt.tight_layout()
plt.show()

## 7. Comparison Protocol

For a fair three-way comparison, primal-dual, sliding-window CUCB, and change-detector CUCB must load the same `realization_XX.npy` path for each run. The matrix fixes every highest competing bid. The associated `config.json` fixes the constellation, sample seed, change points, and mode schedule. Algorithm-specific random seeds may remain separate and must be recorded in the final output.